# Data Preparation for Instruction Fine-Tuning

Preparing clean instruction-response dataset for fine-tuning a language model **Mistral-7B-Instruct**. I use the `databricks/databricks-dolly-15k` dataset from Hugging Face.

## Steps
1. Dataset loading
2. Cleaning and filtration
3. Remove duplicates and dataset collection
4. Saving as JSONL

## Imports

In [2]:
from datasets import load_dataset
from tqdm import tqdm
import json

## Dataset Loading

In [4]:
dataset = load_dataset("databricks/databricks-dolly-15k")
data = dataset["train"]
print(f"Total samples: {len(data)}")
data[0]

Total samples: 15011


{'instruction': 'When did Virgin Australia start operating?',
 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.",
 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.',
 'category': 'closed_qa'}

## Cleaning and Filtration

In [5]:
def is_valid(example):
    instruction = example["instruction"].strip()
    response = example["response"].strip()

    if not instruction or not response:
        return False

    if len(response.split()) < 10:
        return False

    if len(response.split()) > 300:
        return False

    return True

## Removing duplicates and collecting dataset

In [7]:
clean_data = []
seen = set()

for example in tqdm(data):
    if not is_valid(example):
        continue

    instruction = example["instruction"].strip()
    response = example["response"].strip()

    key = (instruction, response)

    if key in seen:
        continue

    seen.add(key)

    clean_data.append({
        "instruction": instruction,
        "response": response
    })

clean_data = clean_data[:300]

def add_style(example):
    example["instruction"] = "Answer clearly and concisely: " + example["instruction"]
    return example

clean_data = [add_style(x) for x in clean_data]

print(f"Final dataset size: {len(clean_data)}")

100%|██████████| 15011/15011 [00:00<00:00, 36572.67it/s]

Final dataset size: 300


## Saving as JSONL

In [8]:
clean_data[0]

{'instruction': 'Answer clearly and concisely: When did Virgin Australia start operating?',
 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.'}

In [9]:
output_path = "../data/dataset.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for item in clean_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved to {output_path}")

Saved to ../data/dataset.jsonl
